---

**Load and Explore the Data**


---


In [1]:
import pandas as pd
import sqlite3

In [14]:
conn = sqlite3.connect('/content/database.sqlite')
cursor = conn.cursor()

In [15]:
def sqlq(query):
    return pd.read_sql_query(query, conn)

In [17]:
# Prints the names of all tables and their columns

def print_all_table_columns():
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    for table in tables:
        print(f"--- {table} ---")
        cursor.execute(f"PRAGMA table_info('{table}')")
        columns = [col[1] for col in cursor.fetchall()]
        print(", ".join(columns))
        print()
    cursor.close()

print_all_table_columns()

--- Player ---
Player_Id, Player_Name, DOB, Batting_hand, Bowling_skill, Country_Name

--- Extra_Runs ---
Match_Id, Over_Id, Ball_Id, Extra_Type_Id, Extra_Runs, Innings_No

--- Batsman_Scored ---
Match_Id, Over_Id, Ball_Id, Runs_Scored, Innings_No

--- Batting_Style ---
Batting_Id, Batting_hand

--- Bowling_Style ---
Bowling_Id, Bowling_skill

--- Country ---
Country_Id, Country_Name

--- Season ---
Season_Id, Man_of_the_Series, Orange_Cap, Purple_Cap, Season_Year

--- City ---
City_Id, City_Name, Country_id

--- Outcome ---
Outcome_Id, Outcome_Type

--- Win_By ---
Win_Id, Win_Type

--- Wicket_Taken ---
Match_Id, Over_Id, Ball_Id, Player_Out, Kind_Out, Fielders, Innings_No

--- Venue ---
Venue_Id, Venue_Name, City_Id

--- Extra_Type ---
Extra_Id, Extra_Name

--- Out_Type ---
Out_Id, Out_Name

--- Toss_Decision ---
Toss_Id, Toss_Name

--- Umpire ---
Umpire_Id, Umpire_Name, Umpire_Country

--- Team ---
Team_Id, Team_Name

--- Ball_by_Ball ---
Match_Id, Over_Id, Ball_Id, Innings_No, Team_

---

**Query 1: Select All Columns from Player’s Table**
* Write and execute a SQL query to select all columns from the Player_Match table.

---

In [5]:
sql = '''
SELECT *
FROM Player_Match
'''
sqlq(sql)

,Match_Id,Player_Id,Role_Id,Team_Id
0,335987,1,1,1
1,335987,2,3,1
2,335987,3,3,1
3,335987,4,3,1
4,335987,5,3,1
...,...,...,...,...
12689,981024,385,3,11
12690,981024,394,3,11
12691,981024,429,3,11
12692,981024,434,3,2


**Query 2: Batsman vs Runs**

* Write and execute a SQL query to calculate the total runs scored by each batsman.

In [6]:
sql = '''
SELECT
    p.Player_Name,
    SUM(bs.Runs_Scored) AS Total_Runs
FROM Batsman_Scored bs
JOIN Ball_by_Ball bbb ON bs.Match_Id = bbb.Match_Id AND bs.Over_Id = bbb.Over_Id AND bs.Ball_Id = bbb.Ball_Id AND bs.Innings_No = bbb.Innings_No
JOIN Player p ON bbb.Striker = p.Player_Id
GROUP BY p.Player_Name
ORDER BY Total_Runs DESC;
'''
sqlq(sql)

,Player_Name,Total_Runs
0,SK Raina,4106
1,V Kohli,4105
2,RG Sharma,3874
3,G Gambhir,3634
4,CH Gayle,3447
...,...,...
429,L Ablish,0
430,IC Pandey,0
431,C Nanda,0
432,Abdur Razzak,0


---

**Query 3: Fifties and Hundreds**

* Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman.

---



In [7]:
sql = '''

SELECT
  p.Player_Name,
  SUM(CASE WHEN inning_runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
  SUM(CASE WHEN inning_runs >= 100 THEN 1 ELSE 0 END) AS hundreds
FROM (
  SELECT
    b.Striker AS Player_Id,
    b.Match_Id,
    b.Innings_No,
    SUM(bs.Runs_Scored) AS inning_runs
  FROM Ball_by_Ball b
  JOIN Batsman_Scored bs
    ON b.Match_Id = bs.Match_Id AND b.Over_Id = bs.Over_Id AND b.Ball_Id = bs.Ball_Id AND b.Innings_No = bs.Innings_No
  GROUP BY b.Striker, b.Match_Id, b.Innings_No
) AS innings_scores
JOIN Player p ON innings_scores.Player_Id = p.Player_Id
GROUP BY p.Player_Name
ORDER BY hundreds DESC, fifties DESC;
'''

sqlq(sql)


,Player_Name,fifties,hundreds
0,CH Gayle,20,5
1,V Kohli,26,4
2,AB de Villiers,21,3
3,DA Warner,32,2
4,V Sehwag,16,2
...,...,...,...
429,Yashpal Singh,0,0
430,Younis Khan,0,0
431,YS Chahal,0,0
432,YV Takawale,0,0


---

**Query 4: Best Bowling Figures**

* Write and execute a SQL query to find the best bowling figures for each bowler.

---

In [9]:
sql = '''

WITH Bowling_Stats AS (
    SELECT
        bbb.Bowler AS Bowler_Id,
        bbb.Match_Id,
        bbb.Innings_No,
        SUM(bs.Runs_Scored) AS runs_conceded,
        COUNT(wt.Player_Out) AS wickets
    FROM Ball_by_Ball bbb
    LEFT JOIN Batsman_Scored bs
        ON bbb.Match_Id = bs.Match_Id
        AND bbb.Over_Id = bs.Over_Id
        AND bbb.Ball_Id = bs.Ball_Id
        AND bbb.Innings_No = bs.Innings_No
    LEFT JOIN Wicket_Taken wt
        ON bbb.Match_Id = wt.Match_Id
        AND bbb.Over_Id = wt.Over_Id
        AND bbb.Ball_Id = wt.Ball_Id
        AND bbb.Innings_No = wt.Innings_No
        AND wt.Player_Out IS NOT NULL
    GROUP BY bbb.Bowler, bbb.Match_Id, bbb.Innings_No
)
SELECT
    p.Player_Name AS Bowler,
    MAX(wickets) AS Best_Wickets,
    MIN(runs_conceded) AS Best_Runs_Conceded
FROM Bowling_Stats bs
JOIN Player p ON bs.Bowler_Id = p.Player_Id
WHERE wickets > 0
GROUP BY bs.Bowler_Id
ORDER BY Best_Wickets DESC, Best_Runs_Conceded ASC;

'''
sqlq(sql)

,Bowler,Best_Wickets,Best_Runs_Conceded
0,AD Russell,6,6
1,Sohail Tanvir,6,7
2,DJG Sammy,6,10
3,A Zampa,6,19
4,RA Jadeja,5,0
...,...,...,...
284,KJ Abbott,1,37
285,B Geeves,1,37
286,AF Milne,1,38
287,TP Sudhindra,1,44


---

**Query 5: Comprehensive Career Metrics**

* Combine all the previous chunks into a single comprehensive query to get detailed career metrics for players.

---

In [12]:
sql = '''

-- 1. Суммируем по каждому иннингу сколько игрок набрал за иннинг
WITH Innings_Scores AS (
    SELECT
        bbb.Striker AS Player_Id,
        bbb.Match_Id,
        bbb.Innings_No,
        SUM(bs.Runs_Scored) AS Innings_Runs
    FROM Ball_by_Ball bbb
    JOIN Batsman_Scored bs
      ON bbb.Match_Id = bs.Match_Id
     AND bbb.Over_Id = bs.Over_Id
     AND bbb.Ball_Id = bs.Ball_Id
     AND bbb.Innings_No = bs.Innings_No
    GROUP BY bbb.Striker, bbb.Match_Id, bbb.Innings_No
),
Career_Runs AS (
    SELECT
        Player_Id,
        COUNT(DISTINCT Match_Id) AS Matches_Played,
        SUM(Innings_Runs) AS Total_Runs,
        COUNT(CASE WHEN Innings_Runs >= 50 AND Innings_Runs < 100 THEN 1 END) AS Fifties,
        COUNT(CASE WHEN Innings_Runs >= 100 THEN 1 END) AS Hundreds
    FROM Innings_Scores
    GROUP BY Player_Id
),
Bowling_Figures AS (
    SELECT
        bbb.Bowler AS Player_Id,
        COUNT(wt.Player_Out) AS Total_Wickets
    FROM Ball_by_Ball bbb
    LEFT JOIN Wicket_Taken wt
      ON bbb.Match_Id = wt.Match_Id
     AND bbb.Over_Id = wt.Over_Id
     AND bbb.Ball_Id = wt.Ball_Id
     AND bbb.Innings_No = wt.Innings_No
    GROUP BY bbb.Bowler
)
SELECT
    p.Player_Name,
    cr.Matches_Played,
    cr.Total_Runs,
    ROUND(1.0 * cr.Total_Runs / NULLIF(cr.Matches_Played, 0), 2) AS Batting_Average,
    cr.Fifties,
    cr.Hundreds,
    bf.Total_Wickets
FROM Player p
LEFT JOIN Career_Runs cr ON p.Player_Id = cr.Player_Id
LEFT JOIN Bowling_Figures bf ON p.Player_Id = bf.Player_Id
ORDER BY cr.Total_Runs DESC
LIMIT 50;

'''

sqlq(sql)

,Player_Name,Matches_Played,Total_Runs,Batting_Average,Fifties,Hundreds,Total_Wickets
0,SK Raina,143,4106,28.71,28,1,29.0
1,V Kohli,131,4105,31.34,26,4,5.0
2,RG Sharma,137,3874,28.28,29,1,16.0
3,G Gambhir,130,3634,27.95,31,0,NaN
4,CH Gayle,91,3447,37.88,20,5,19.0
5,RV Uthappa,130,3390,26.08,17,0,NaN
6,DA Warner,100,3373,33.73,32,2,NaN
7,MS Dhoni,128,3270,25.55,16,0,NaN
8,AB de Villiers,109,3270,30.00,21,3,NaN
9,S Dhawan,112,3082,27.52,25,0,4.0
